# NB06 — CCR: clinical prior vs. shuffled/uniform controls

**Goal.** Prove that the CCR gain comes from the *clinical* structure of the
co-occurrence matrix `C`, not from any output coupling. We compare three matrices
at the same `λ`, same folds, same seeds:

- **real** — `C[i][j] = P(j=1 | i=1)` estimated per fold from train (the paper's CCR).
- **shuffled** — the off-diagonal entries of the real `C` randomly permuted per fold
  (same value distribution, clinical structure destroyed).
- **uniform** — every off-diagonal entry set to the mean of the real off-diagonals
  (coupling with zero structure).

If **real > shuffled ≈ uniform**, the prior's value is clinical. This is the
central control for the paper's main claim.

> By default we **reuse** the already-trained `m0_swin` (λ=0) and `m2_l06` (real CCR)
> from NB03's `results/m2/m2_results.json`, and only train the two new control
> conditions. Set `RERUN_REAL=True` to retrain everything self-contained.

## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [2]:
import json
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score, roc_auc_score)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'timm    : {timm.__version__}')

## 1. Configuration

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # O caminho correto baseado na sua imagem:
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens para o disco rápido do Colab...')
        # Arrumando o unzip também para o caminho correto da imagem:
        !unzip -q /content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/anonymous-V3/Trilha3-V2')
    IMGS_DIR = Path('e:/anonymous-V3/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'nb06'
NB03_RESULTS = BASE_DIR / 'results' / 'm2' / 'm2_results.json'   # reuse real CCR + M0
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'

for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_FOLDS     = 5
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE     = 224
BATCH_SIZE   = 128          # 96 GB VRAM: pode subir p/ 256-512 (ver nota abaixo)
MAX_EPOCHS   = 50
LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
ES_PATIENCE  = 10
LR_PATIENCE  = 4
LR_FACTOR    = 0.5
SEEDS        = [42, 43, 44, 45, 46]
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CCR controlado — λ fixo no melhor valor do paper (NB03)
LAMBDA   = 0.6
RERUN_REAL = False     # True = re-treina 'real' do zero; False = reusa NB03 m2_l06

# Condições de controle a TREINAR neste notebook
CONTROL_CONDITIONS = ['shuffled', 'uniform']
if RERUN_REAL:
    CONTROL_CONDITIONS = ['real'] + CONTROL_CONDITIONS

# NOTA p/ GPU grande (G4 96 GB): aumentar BATCH_SIZE acelera muito.
# Se subir o batch, considere escalar LR proporcional (linear scaling rule).
# As 3 condições usam o MESMO batch, então a comparação interna não é afetada.

print(f'Device     : {DEVICE}')
print(f'λ fixo     : {LAMBDA}')
print(f'Condições  : {CONTROL_CONDITIONS}')
print(f'Reusa real : {not RERUN_REAL} (de {NB03_RESULTS.name})')

## 2. Dataset, transforms and helpers (same as NB03/NB04)

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str) -> T.Compose:
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
            T.CenterCrop(IMG_SIZE),
            T.ToTensor(),
            normalize,
        ])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.imgs_dir  = imgs_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def compute_pos_weights(df) -> torch.Tensor:
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    return torch.tensor(neg / (pos + 1e-6), dtype=torch.float32).to(DEVICE)

def compute_cooccurrence_matrix(df) -> np.ndarray:
    """C[i][j] = P(label_j=1 | label_i=1), por fold (sem leakage). Retorna np.ndarray."""
    labels = df[CORE_LABELS].values.astype(float)
    C = np.zeros((N_CLASSES, N_CLASSES))
    for i in range(N_CLASSES):
        mask_i = labels[:, i] == 1
        n_i    = mask_i.sum()
        if n_i == 0:
            continue
        for j in range(N_CLASSES):
            if i == j:
                continue
            C[i][j] = labels[mask_i, j].sum() / n_i
    return C

print('Dataset e funções auxiliares definidos.')

## 3. Building the three co-occurrence matrices

All three share the same diagonal (zero) and the same set of off-diagonal *values*;
only the **arrangement** differs. The shuffle is deterministic per fold (seeded), so
the notebook is fully reproducible and resumable.

In [5]:
def make_C_variant(C_real: np.ndarray, variant: str, fold: int) -> torch.Tensor:
    """
    real     → matriz clínica original.
    shuffled → permuta as entradas off-diagonal (destrói estrutura, mantém valores).
    uniform  → todas as off-diagonais = média das off-diagonais reais.
    """
    C = np.zeros_like(C_real)
    off_mask = ~np.eye(N_CLASSES, dtype=bool)
    off_vals = C_real[off_mask]

    if variant == 'real':
        C = C_real.copy()
    elif variant == 'shuffled':
        # permutação determinística por fold — não usa estado global do numpy
        rng = np.random.default_rng(1000 + fold)
        perm = rng.permutation(off_vals)
        C[off_mask] = perm
    elif variant == 'uniform':
        C[off_mask] = off_vals.mean()
    else:
        raise ValueError(f'Variante desconhecida: {variant}')
    return torch.tensor(C, dtype=torch.float32).to(DEVICE)


class M2CoocLoss(nn.Module):
    """L = L_BCE + λ * Σ_{i≠j} C[i][j] * relu(p_i - p_j)²  (mesma do NB03)."""
    def __init__(self, pos_weight, cooc_matrix, lam):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.C   = cooc_matrix
        self.lam = lam
    def forward(self, logits, targets):
        loss_bce = self.bce(logits, targets)
        if self.lam == 0.0:
            return loss_bce
        probs = torch.sigmoid(logits)
        p_i   = probs.unsqueeze(2)
        p_j   = probs.unsqueeze(1)
        diff  = torch.relu(p_i - p_j)
        loss_coo = (self.C.unsqueeze(0) * diff.pow(2)).mean()
        return loss_bce + self.lam * loss_coo

# Sanidade: as 3 variantes preservam soma das off-diagonais (real e shuffled idênticas)
_df0 = pd.read_csv(SPLITS_DIR / 'fold_0_train.csv')
_Creal = compute_cooccurrence_matrix(_df0)
for v in ['real', 'shuffled', 'uniform']:
    _Cv = make_C_variant(_Creal, v, 0).cpu().numpy()
    print(f'  {v:<9}: soma off-diag={_Cv[~np.eye(N_CLASSES,dtype=bool)].sum():.3f}  '
          f'(real={_Creal[~np.eye(N_CLASSES,dtype=bool)].sum():.3f})')
print('make_C_variant e M2CoocLoss definidas.')

## 4. Model, optimizer and metrics (same as NB03/NB04)

In [6]:
def build_swin_tiny(n_classes: int = N_CLASSES) -> nn.Module:
    return timm.create_model(
        'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
        pretrained=True, num_classes=n_classes)

def build_optimizer(model: nn.Module):
    head_params     = list(model.head.parameters())
    backbone_params = [p for n, p in model.named_parameters() if 'head' not in n]
    return torch.optim.AdamW(
        [{'params': backbone_params, 'lr': LR_BACKBONE},
         {'params': head_params,     'lr': LR_HEAD}],
        weight_decay=WEIGHT_DECAY)

def optimize_thresholds(probs, labels):
    thresholds = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0:
                continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t
    return thresholds

def compute_metrics(probs, labels, thresholds=None):
    if thresholds is None:
        thresholds = np.full(N_CLASSES, 0.5)
    preds = (probs >= thresholds).astype(int)
    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    pr_per = precision_score(labels, preds, average=None, zero_division=0)
    rc_per = recall_score(labels, preds, average=None, zero_division=0)
    auprc, rocauc = [], []
    for i in range(N_CLASSES):
        auprc.append(average_precision_score(labels[:, i], probs[:, i])
                     if labels[:, i].sum() > 0 else float('nan'))
        rocauc.append(roc_auc_score(labels[:, i], probs[:, i])
                      if 0 < labels[:, i].sum() < len(labels) else float('nan'))
    multi_mask = labels.sum(axis=1) >= 2
    f1_multi = float(f1_score(labels[multi_mask], preds[multi_mask],
                              average='macro', zero_division=0)) if multi_mask.sum() > 0 else float('nan')
    result = {
        'f1_macro': float(np.nanmean(f1_per)),
        'f1_micro': float(f1_score(labels, preds, average='micro', zero_division=0)),
        'hamming':  float(np.mean(preds != labels)),
        'f1_multi': f1_multi,
        'pr_auc_macro':  float(np.nanmean(auprc)),
        'roc_auc_macro': float(np.nanmean(rocauc)),
    }
    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']     = float(f1_per[i])
        result[f'prec_{label}']   = float(pr_per[i])
        result[f'rec_{label}']    = float(rc_per[i])
        result[f'auprc_{label}']  = float(auprc[i])
        result[f'rocauc_{label}'] = float(rocauc[i])
        result[f'thr_{label}']    = float(thresholds[i])
    return result

print('Modelo, otimizador e métricas definidos.')

## 5. Training loop (one fold, one C-variant)

In [7]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = True

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def train_one_fold(fold, variant, save_checkpoint=True, verbose=True):
    set_seed(SEEDS[fold])
    df_tr  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv')
    df_val = pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv')
    df_te  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')

    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, get_transforms('train'))
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, get_transforms('val'))
    ds_te  = GastroscopyDataset(df_te,  IMGS_DIR, get_transforms('val'))
    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model     = build_swin_tiny().to(DEVICE)
    pos_w     = compute_pos_weights(df_tr)
    C_real    = compute_cooccurrence_matrix(df_tr)
    C_variant = make_C_variant(C_real, variant, fold)
    criterion = M2CoocLoss(pos_w, C_variant, LAMBDA)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                           patience=LR_PATIENCE, factor=LR_FACTOR)
    best_val_f1, best_weights, es_counter = -1.0, None, 0
    history = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(imgs)
        train_loss = running_loss / len(ds_tr)
        val_probs, val_labels = evaluate(model, dl_val)
        val_thr = optimize_thresholds(val_probs, val_labels)
        val_f1  = compute_metrics(val_probs, val_labels, val_thr)['f1_macro']
        scheduler.step(val_f1)
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_f1)
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={val_f1:.4f}')
        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1, best_weights, es_counter = val_f1, deepcopy(model.state_dict()), 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                if verbose: print(f'  Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        torch.save(best_weights, CKPT_DIR / f'swin_ccr_{variant}_fold{fold}.pt')
    test_probs, test_labels = evaluate(model, dl_te)
    test_met = compute_metrics(test_probs, test_labels, val_thr)
    test_met.update({
        'fold': fold, 'variant': variant, 'lam': LAMBDA,
        'best_val_f1': best_val_f1, 'epochs_trained': len(history['train_loss']),
        'val_thresholds': val_thr.tolist(), 'history': history,
        'test_preds':  (test_probs >= val_thr).astype(int).tolist(),
        'test_labels': test_labels.tolist(),
    })
    return test_met

print('Loop de treino definido.')

## 6. Execution — train control conditions (with resume)

In [8]:
RUN_FAST_CHECK = False
if RUN_FAST_CHECK:
    MAX_EPOCHS, FOLDS_TO_RUN = 5, [0]
    print('MODO FAST CHECK: 1 fold, 5 épocas.')
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))
    print(f'MODO COMPLETO: {len(CONTROL_CONDITIONS)} condições × {N_FOLDS} folds = '
          f'{len(CONTROL_CONDITIONS) * N_FOLDS} runs novos.')

results_file = RESULTS_DIR / 'ccr_control_results.json'
if results_file.exists():
    print('>> ENCONTRADO RESULTADO ANTERIOR! Retomando... <<')
    with open(results_file, encoding='utf-8') as f:
        all_results = json.load(f)
else:
    all_results = {}

# Reusa M0 e CCR real do NB03 (a menos que RERUN_REAL)
if not RERUN_REAL and 'real' not in all_results:
    with open(NB03_RESULTS, encoding='utf-8') as f:
        nb03 = json.load(f)
    for src_key, dst_key in [('m0_swin', 'base_l0'), ('m2_l06', 'real')]:
        if src_key in nb03:
            all_results[dst_key] = {
                'config': {'variant': dst_key, 'lam': 0.0 if dst_key == 'base_l0' else LAMBDA,
                           'source': f'NB03:{src_key}'},
                'fold_results': nb03[src_key]['fold_results'],
                'aggregated':   nb03[src_key]['aggregated'],
            }
    print(f"Reusados do NB03: base_l0 (M0) e real (m2_l06).")

for variant in CONTROL_CONDITIONS:
    if variant in all_results and len(all_results[variant].get('fold_results', [])) >= len(FOLDS_TO_RUN):
        print(f'\n[PULANDO] Condição {variant} já concluída.')
        continue
        
    print(f'\n{"="*60}\nCondição CCR: {variant}  (λ={LAMBDA})\n{"="*60}')
    
    # Se a variante for nova, inicializa a estrutura
    if variant not in all_results:
        all_results[variant] = {'config': {'variant': variant, 'lam': LAMBDA}, 'fold_results': []}
        
    # Carrega folds já terminados desta variante (caso tenha caído a conexão)
    fold_results = all_results[variant].get('fold_results', [])
    completed_folds = {r['fold'] for r in fold_results}
    t0 = time.time()
    
    for fold in FOLDS_TO_RUN:
        if fold in completed_folds:
            print(f'\n  Fold {fold}/{N_FOLDS-1} já concluído, pulando.')
            continue
            
        print(f'\n  Fold {fold}/{N_FOLDS-1}')
        met = train_one_fold(fold, variant, save_checkpoint=not RUN_FAST_CHECK)
        fold_results.append(met)
        print(f'  → Test Macro-F1={met["f1_macro"]:.4f}  PÓLIPO={met["f1_PÓLIPO"]:.4f}  '
              f'MICRONOD={met["f1_MICRONODULARIDADE"]:.4f}')
              
        # SALVAMENTO INTERMEDIÁRIO: Salva no JSON imediatamente após cada fold
        all_results[variant]['fold_results'] = fold_results
        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
            
    elapsed = time.time() - t0
    
    # Após todos os folds terminarem, calcula os agregados
    metric_keys = (['f1_macro','f1_micro','hamming','f1_multi','pr_auc_macro','roc_auc_macro'] +
                   [f'f1_{l}' for l in CORE_LABELS] + [f'auprc_{l}' for l in CORE_LABELS])
    agg = {}
    for k in metric_keys:
        vals = [r[k] for r in fold_results if not np.isnan(r.get(k, float('nan')))]
        if vals:
            agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
            
    all_results[variant]['aggregated'] = agg
    # Soma o tempo caso tenha sido retomado
    all_results[variant]['elapsed_sec'] = round(all_results[variant].get('elapsed_sec', 0) + elapsed, 1)
    
    print(f'\n  AGREGADO: Macro-F1 = {agg["f1_macro"]["mean"]:.4f} ± {agg["f1_macro"]["std"]:.4f}')
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print('  Salvo agregado da condição.')

## 7. Statistical test — real vs. shuffled vs. uniform

Paired bootstrap (B=2000) on the pooled 5-fold test predictions. The key
comparison is **real − shuffled**: if its 95% CI excludes zero, the clinical
structure (not mere coupling) drives the gain.

In [9]:
def collect(variant):
    all_p, all_l = [], []
    for r in all_results[variant]['fold_results']:
        all_p.extend(r['test_preds']); all_l.extend(r['test_labels'])
    return np.array(all_p), np.array(all_l)

def bootstrap_compare(preds_a, labels_a, preds_b, labels_b, B=2000, seed=42):
    """Δ = F1(b) − F1(a), pareado. p = P(Δ ≤ 0)."""
    rng = np.random.default_rng(seed)
    n = len(labels_a)
    deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        f1_a = f1_score(labels_a[idx], preds_a[idx], average='macro', zero_division=0)
        f1_b = f1_score(labels_b[idx], preds_b[idx], average='macro', zero_division=0)
        deltas.append(f1_b - f1_a)
    deltas = np.array(deltas)
    return {'delta_mean': float(deltas.mean()),
            'ci_95_low': float(np.percentile(deltas, 2.5)),
            'ci_95_high': float(np.percentile(deltas, 97.5)),
            'p_value': float(np.mean(deltas <= 0)),
            'significant': bool(np.percentile(deltas, 2.5) > 0)}

def f1_of(variant):
    p, l = collect(variant)
    return f1_score(l, p, average='macro', zero_division=0)

present = [v for v in ['base_l0', 'real', 'shuffled', 'uniform'] if v in all_results]
print('F1-macro agrupado (5 folds):')
for v in present:
    print(f'  {v:<10}: {f1_of(v):.4f}')
print()

comparisons = {}
# Comparações-chave: real vs cada controle, e cada um vs base_l0
pairs = []
if 'real' in all_results:
    for ctrl in ['shuffled', 'uniform']:
        if ctrl in all_results:
            pairs.append((ctrl, 'real'))   # Δ = real − ctrl (esperado > 0)
if 'base_l0' in all_results:
    for v in ['real', 'shuffled', 'uniform']:
        if v in all_results:
            pairs.append(('base_l0', v))   # Δ = v − base

print(f'{"Comparação":<24} {"Δ F1":>9} {"CI95":>20} {"p":>8} {"Sig":>5}')
print('-'*70)
for a, b in pairs:
    pa, la = collect(a); pb, lb = collect(b)
    boot = bootstrap_compare(pa, la, pb, lb)
    key = f'{b}_vs_{a}'
    comparisons[key] = boot
    ci = f'[{boot["ci_95_low"]:+.3f}, {boot["ci_95_high"]:+.3f}]'
    sig = '✓' if boot['significant'] else '—'
    print(f'{key:<24} {boot["delta_mean"]:>+9.4f} {ci:>20} {boot["p_value"]:>8.4f} {sig:>5}')

with open(RESULTS_DIR / 'ccr_control_bootstrap.json', 'w', encoding='utf-8') as f:
    json.dump(comparisons, f, ensure_ascii=False, indent=2)

# Veredito automático da tese
if 'real_vs_shuffled' in comparisons:
    rs = comparisons['real_vs_shuffled']
    print('\n=== VEREDITO ===')
    if rs['significant']:
        print(f'✓ CCR real > shuffled (Δ={rs["delta_mean"]:+.4f}, CI exclui 0). '
              'O prior CLÍNICO tem valor — não é mero acoplamento.')
    else:
        print(f'⚠ real vs shuffled NÃO significativo (Δ={rs["delta_mean"]:+.4f}, '
              f'CI=[{rs["ci_95_low"]:+.3f},{rs["ci_95_high"]:+.3f}]). '
              'Reportar honestamente: o ganho pode vir do acoplamento, não da estrutura clínica.')
print('\nComparações salvas.')

## 8. Figures (EN + PT)

In [10]:
ORDER = [v for v in ['base_l0', 'shuffled', 'uniform', 'real'] if v in all_results]
LABELS = {
    'pt': {'base_l0': 'Base\n(λ=0)', 'shuffled': 'CCR\nshuffled',
           'uniform': 'CCR\nuniform', 'real': 'CCR\nreal'},
    'en': {'base_l0': 'Base\n(λ=0)', 'shuffled': 'CCR\nshuffled',
           'uniform': 'CCR\nuniform', 'real': 'CCR\nreal'},
}
TITLES = {
    'pt': ('CCR: prior clínico vs. controles (Swin-Tiny, λ=0.6)',
           'Macro-F1 (média ± dp, 5 folds)', 'F1 por classe × condição'),
    'en': ('CCR: clinical prior vs. controls (Swin-Tiny, λ=0.6)',
           'Macro-F1 (mean ± sd, 5 folds)', 'F1 per class × condition'),
}
COLORS = {'base_l0': '#90A4AE', 'shuffled': '#E65100', 'uniform': '#FBC02D', 'real': '#2E7D32'}

def make_figure(lang):
    means = [all_results[v]['aggregated']['f1_macro']['mean'] for v in ORDER]
    stds  = [all_results[v]['aggregated']['f1_macro']['std']  for v in ORDER]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    bars = ax.bar([LABELS[lang][v] for v in ORDER], means, yerr=stds,
                  color=[COLORS[v] for v in ORDER], capsize=5,
                  edgecolor='white', linewidth=0.6)
    if 'base_l0' in all_results:
        ax.axhline(all_results['base_l0']['aggregated']['f1_macro']['mean'],
                   color='gray', linestyle='--', linewidth=1, alpha=0.6)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, m+0.005, f'{m:.3f}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylabel(TITLES[lang][1], fontsize=10)
    ax.set_title(TITLES[lang][0], fontweight='bold', fontsize=11)
    ax.set_ylim(min(means)-0.04, max(means)+0.04)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    heat = {LABELS[lang][v].replace(chr(10), ' '):
            [all_results[v]['aggregated'].get(f'f1_{l}', {}).get('mean', float('nan'))
             for l in CORE_LABELS] for v in ORDER}
    df_heat = pd.DataFrame(heat, index=CORE_LABELS).T
    sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.3, vmax=0.85, linewidths=0.4, ax=ax, cbar_kws={'label': 'F1'})
    ax.set_title(TITLES[lang][2], fontweight='bold')
    plt.tight_layout()
    out = FIGS_DIR / f'nb06_ccr_control_{lang}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura salva: {out.name}')

for lang in ['en', 'pt']:
    make_figure(lang)

## 9. Comparative table

In [11]:
rows = []
NAMES = {'base_l0': 'Base BCE (λ=0)', 'shuffled': 'CCR shuffled (λ=0.6)',
         'uniform': 'CCR uniform (λ=0.6)', 'real': 'CCR real (λ=0.6)'}
for v in [x for x in ['base_l0', 'shuffled', 'uniform', 'real'] if x in all_results]:
    agg = all_results[v]['aggregated']
    row = {'Condition': NAMES[v]}
    for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO', 'f1_ÚLCERA', 'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
        val = agg.get(k)
        row[k.replace('f1_', '')] = f"{val['mean']:.3f}±{val['std']:.3f}" if val else '-'
    rows.append(row)
df_table = pd.DataFrame(rows).set_index('Condition')
print('=== CCR CONTROL — COMPARATIVE TABLE ===')
print(df_table.to_string())
df_table.to_csv(RESULTS_DIR / 'tabela_nb06.csv')
print('\nTabela salva em results/nb06/tabela_nb06.csv')